In [ ]:
import rospy
from std_msgs.msg import String, Int32, Float32
import sensor_msgs.msg
import math
import random
import numpy as np
from geometry_msgs.msg import Twist
from itertools import *
from operator import itemgetter

THRESHOLD = 0.3 #THRESHOLD alowed distance to obstacle
turn_angle = Float32()
komunikat = String()

def LaserScanProcess(data):
    global turn_angle
    global pub_kom
    global angle
    range_angels = np.arange(len(data.ranges))
    ranges = np.array(data.ranges)
    range_mask = (ranges > THRESHOLD)
    ranges = list(range_angels[range_mask])
    
    gap_list = []       
    for k, g in groupby(enumerate(ranges), lambda x:x[0]-x[1]):
        gap_list.append(np.fromiter(map(itemgetter(1), g), dtype=np.int))
    gap_elem = len(gap_list)
    
    if gap_elem == 0:
    #Nie istnieje dopuszczalny kierunek ruchu, robot jest calkowicie zastawiony
        turn_angle = -1
        komunikat.data = "brak mozliwych ruchow"
    elif gap_elem == 1:
    #Tylko jeden obszar z dopuszczalnym kierunkiem ruchu    
        
        if gap_list[0][0] == 0 and gap_list[0][-1] == 359:
            komunikat.data = "dostepna cala przestrzen ruchu"
            turn_angle = 0
        else:
            komunikat.data = "dostepny jeden niepelny obszar ruchu"
            min_angle = gap_list[0][0]
            max_angle = gap_list[0][-1]
            turn_angle = (min_angle + max_angle)/2
        
    else:
    #Sprawdzenie czy lewy i prawy obszar jednoczesnie istnieja 
        if gap_list[0][0] == 0 and gap_list[-1][-1] == 359:
        #analiza danych gdy oba obszary istnieja
            komunikat.data = "dostepny prawy i lewy obszar"
            left_range_len = len(gap_list[0])
            right_range_len = len(gap_list[-1])
            
            sorted_list = gap_list[:]
            sorted_list.sort(key=len)
            
            if left_range_len + right_range_len > len(sorted_list[-1]):
            #Laczony obszar lewy i prawy
                #right_mid_angle = (gap_list[-1][0]+gap_list[-1][-1])/2
                #left_mid_angle = (gap_list[0][0]+gap_list[0][-1])/2
                right_mid_angle = (gap_list[-1][-1]-gap_list[-1][0])/2
                left_mid_angle = (gap_list[0][-1]-gap_list[0][0])/2
                
                if left_range_len > right_range_len:
                    komunikat.data = " lewy obszar wiekszy niz prawy " 
                    turn_angle = gap_list[0][-1] - left_mid_angle - right_mid_angle
                else:
                    komunikat.data = "prawy obszar wiekszy niz lewy"
                    turn_angle = gap_list[-1][0]%360 + right_mid_angle + left_mid_angle     
            else:
            #wiekszy jest obszar nielaczony
                komunikat.data = "najwiekszy jest obszar nielaczony"
                min_angle = sorted_list[-1][0]
                max_angle = sorted_list[-1][-1]
                turn_angle = (min_angle + max_angle)/2   
        else:
        #Jednoczesnie istnieje tylko lewy lub prawy obszar
            komunikat.data = "istnieje tylko lewy lub prawy obszar"
            gap_list.sort(key=len)
            largest_gap = gap_list[-1]
            min_angle = largest_gap[0]
            max_angle = largest_gap[-1]
            turn_angle = (min_angle + max_angle)/2
       
    turn_angle = turn_angle%360
    if turn_angle > 180:
        #Konwersja kata na zakres od 0 do 180 - od 0 do -180
        #Wartosci dotatnie oznaczaja ze robot powinien obrocic sie zgodnie ze wskazowkami zegara w dana strone
        #wartosci ujemne - robot powinien obrocic sie w prawa strone o dany kat
        turn_angle = turn_angle - 360.0

In [ ]:
if __name__ == '__main__':
    global turn_angle
    global komunikat
    rospy.init_node('listener', anonymous=True)
    rospy.Subscriber("scan", sensor_msgs.msg.LaserScan , LaserScanProcess)
    pub_kom = rospy.Publisher('/komunikat', String, queue_size=10)
    pub_angle = rospy.Publisher('/turn_angle', Float32, queue_size=10)
    
    rate = rospy.Rate(10) # 10hz
    while not rospy.is_shutdown():
        pub_angle.publish(turn_angle)
        pub_kom.publish(komunikat)
        rate.sleep()